In [1]:
# Must precede any shapiq import (incl. transitively via Benchmarking.backends
# below), or a later xgboost/lightgbm .fit() segfaults — see run_benchmark.py.
import xgboost  # noqa: F401
import lightgbm  # noqa: F401

# Tree-specific benchmark

Mirrors `slurm/run_benchmark.py`'s tree sweep against `configs/config-tree.yaml`:
tree-only true-value backends for xgboost/lightgbm models plus order-2
interactions. Reuses `TREE_TRUE_VALUE_MAP`/`INTERACTION_TRUE_VALUE_MAP` from
`slurm/run_benchmark.py` so both entry points stay in sync.

In [2]:
# Auto-reload edited modules (Benchmarking/, Models/, ...) before each cell runs,
# so editing a .py file takes effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [3]:
# Imports
import warnings
import yaml
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import ParameterGrid

from Models.config_parser import load_config, load_dataset_config, load_seed, as_list
from Models.dataset_and_models import Dataset, Model
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import ShapTrueValueBackend, ShapInteractionBackend
from slurm.run_benchmark import TREE_TRUE_VALUE_MAP, INTERACTION_TRUE_VALUE_MAP

warnings.filterwarnings(
    "ignore",
    message="Not all budget is required due to the border-trick.",
    category=UserWarning,
    module="shapiq",
)
warnings.filterwarnings(
    "ignore",
    message="The sample size is larger than the number of data points in the background set.*",
    category=UserWarning,
    module="shapiq",
)

/Users/itamarfink/Library/CloudStorage/OneDrive-Personal/Documents/University/LMU/SS26/Trustworthy AI/Benchmarker/PR_ModeAgnostic/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/itamarfink/Library/CloudStorage/OneDrive-Personal/Documents/University/LMU/SS26/Trustworthy AI/Benchmarker/PR_ModeAgnostic/.venv/lib/python3.12/site-packages/dalex/_global_checks.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Choose the config to run — required, no default (mirrors slurm/submit.sh, which
# also takes the config explicitly). Tree sweeps expect a tree config; set CONFIG to
# one of the files in ../configs/.
import glob

CONFIG = None  # <-- set this, e.g. "../configs/config-tree.yaml"

if CONFIG is None:
    _available = "\n  ".join(sorted(glob.glob("../configs/*.yaml")))
    raise ValueError(f"Set CONFIG to one of the available configs:\n  {_available}")

model_config = load_config(CONFIG)
dataset_config = load_dataset_config(CONFIG)
seed = load_seed(CONFIG)

model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]

print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = "
      f"{len(model_runs) * len(dataset_runs)} (model, dataset) cells")

2 model configs × 9 dataset configs = 18 (model, dataset) cells


In [ ]:
with open(CONFIG) as f:
    bench = yaml.safe_load(f)["benchmark"]

# seed and n_background may each be a scalar or a list; as_list normalizes both so we
# sweep them as extra loop dimensions, exactly like slurm/run_benchmark.py.
seeds = as_list(bench["seed"])
n_backgrounds = as_list(bench["n_background"])
n_eval = bench["n_eval"]
imputer = bench["imputer"]
tree_libraries = bench.get("tree_libraries", [])
tree_modes = bench.get("tree_modes", [])
interaction_libs = bench.get("interaction_libraries", [])
interaction_max_features = bench.get("interaction_max_features", 16)

OUTPUT_CSV = "../Benchmarking/results_config-tree.csv"

total_weight = len(seeds) * len(n_backgrounds) * sum(
    dataset_params.get("n_features", 1)
    for _, dataset_params in dataset_runs
    for _ in model_runs
)

with tqdm(total=total_weight, desc="tree benchmark", unit="feat") as bar:
    for seed in seeds:
        for dataset_key, dataset_params in dataset_runs:
            dataset_enum = Dataset[dataset_key.upper()]
            weight = dataset_params.get("n_features", 1)
            ds = dataset_enum.load_dataset(**dataset_params, seed=seed)
            X, y = ds["X"], ds["y"]

            for model_key, model_params in model_runs:
                model_enum = Model[model_key.upper()]
                trainer = model_enum.get_model_with_params(dataset_enum, model_params, seed=seed)
                trainer.fit(X, y, task=ds["task"])
                model = trainer.get_model()

                # ShapTrueValueBackend must stay first: it's picked as the oracle.
                true_value_backends = [ShapTrueValueBackend]
                if model_enum.is_tree:
                    for lib in tree_libraries:
                        for mode in tree_modes:
                            cls = TREE_TRUE_VALUE_MAP.get((lib, mode))
                            if cls is not None:
                                true_value_backends.append(cls)

                for n_background in n_backgrounds:
                    bar.set_postfix_str(
                        f"{dataset_key} nf={dataset_params.get('n_features')} | {model_key} "
                        f"| seed={seed} nbg={n_background}"
                    )
                    runner = BenchmarkRunner(
                        true_value_backends=true_value_backends,
                        approximation_specs=[],
                        output_csv=OUTPUT_CSV,
                        n_background=n_background,
                        n_eval=n_eval,
                        seed=seed,
                        imputer=imputer,
                    )
                    runner.run(
                        model=model,
                        X=X,
                        run_meta={
                            "dataset": dataset_key,
                            "model": model_key,
                            "n_features": dataset_params.get("n_features"),
                            "n_samples": dataset_params.get("n_samples"),
                            "order": 1,
                            "n_background": n_background,
                        },
                    )

                    # Pairwise interactions: a separate runner.run() call (different oracle,
                    # different output shape) writing to the same output CSV.
                    if model_enum.is_tree and interaction_libs and X.shape[1] <= interaction_max_features:
                        # ShapInteractionBackend must stay first: it's the order-2 oracle.
                        interaction_backends = [ShapInteractionBackend]
                        for lib in interaction_libs:
                            cls = INTERACTION_TRUE_VALUE_MAP.get(lib)
                            if cls is not None:
                                interaction_backends.append(cls)

                        interaction_runner = BenchmarkRunner(
                            true_value_backends=interaction_backends,
                            approximation_specs=[],
                            output_csv=OUTPUT_CSV,
                            n_background=n_background,
                            n_eval=n_eval,
                            seed=seed,
                            imputer=imputer,
                        )
                        interaction_runner.run(
                            model=model,
                            X=X,
                            run_meta={
                                "dataset": dataset_key,
                                "model": model_key,
                                "n_features": dataset_params.get("n_features"),
                                "n_samples": dataset_params.get("n_samples"),
                                "order": 2,
                                "n_background": n_background,
                            },
                        )
                    bar.update(weight)

print(f"Done. Tree results written to {OUTPUT_CSV}")

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:00<00:00, 205.42it/s]


M time: 0.0 sec, s time: 0.11 sec (3887 _get_s_vectors_given_f calls, f prepare time: 0.038396358489990234, f mean non zero size: 6.557756624646257)


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:00<00:00, 119.67it/s]


M time: 0.0 sec, s time: 0.15 sec (3887 _get_s_vectors_given_f calls, f prepare time: 0.04860258102416992, f mean non zero size: 5.027784924105994)
  [SKIP] fasttreeshap_path_dependent: fasttreeshap 0.1.6 cannot parse XGBoost 3.x's model format (confirmed upstream incompatibility)


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:00<00:00, 232.90it/s]


...

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:00<00:00, 178.84it/s]


M time: 0.01 sec, s time: 0.18 sec (3100 _get_s_vectors_given_f calls, f prepare time: 0.05976223945617676, f mean non zero size: 62.75677419354839)


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:00<00:00, 159.44it/s]


M time: 0.01 sec, s time: 0.17 sec (3100 _get_s_vectors_given_f calls, f prepare time: 0.058283090591430664, f mean non zero size: 36.54838709677419)


tree benchmark:  36%|███▌      | 290/802 [29:42<1:25:52, 10.06s/feat, gisette nf=32 | lightgbm]

In [ ]:
results = pd.read_csv(OUTPUT_CSV)
n_before = len(results)

# Re-running re-emits the oracle row per cell; collapse to one row per
# (cell, seed, n_background, backend, order).
results = results.drop_duplicates(
    subset=["dataset", "model", "n_features", "n_samples", "seed", "n_background",
            "backend", "order"],
    keep="last",
)
results.to_csv(OUTPUT_CSV, index=False)
print(f"de-duplicated {n_before} -> {len(results)} rows; overwrote {OUTPUT_CSV}")

# pairwise_metrics holds a dict of metrics for every (candidate, reference) backend
# pair; pull out the metrics vs. each row's order-specific oracle.
import json

ORACLE_BY_ORDER = {1: "shap_true_value", 2: "shap_interaction"}

def oracle_metric(row, key):
    oracle = ORACLE_BY_ORDER[row["order"]]
    return json.loads(row["pairwise_metrics"]).get(oracle, {}).get(key)

for metric in ["mean_abs_diff", "sign_agreement", "mean_sample_rho"]:
    results[metric] = results.apply(lambda row: oracle_metric(row, metric), axis=1)

cols = [
    "dataset", "model", "n_features", "order", "library", "backend",
    "runtime_s", "mean_abs_diff", "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)